# Gemma 4 — Operational Support vs Operational + Semantic Assistance
## 12 fresh evaluation issues × 2 conditions × 2 repetitions (+ 2 excluded preflight issues)

This is a **targeted diagnostic experiment**, not another generic prompt tweak.

### Primary question
Holding the operational workflow constant, does supplying a frozen, human-structured behavioral specification improve repair correctness?

### Conditions
- **O — operational control:** exact same operationally supported agent + neutral issue restatement + generic review guidance.
- **O+S — semantic assistance:** exact same agent + same neutral issue restatement + issue-specific required behavior, edge cases, and preserved invariants.

### Design safeguards
- Fresh task IDs: excludes every task used in the preserved 416-trajectory corpus.
- Prefer repositories not used previously when the official catalog permits.
- 2 preflight tasks are selected and excluded from evaluation.
- 12 evaluation tasks × 2 conditions × 2 repetitions = **48 evaluation runs**.
- 4 additional preflight runs (2 tasks × 2 conditions).
- Condition order is deterministically randomized within task/repetition.
- Semantic checklists must be completed and frozen **before any model run**.
- No reference patches, hidden evaluator tests, or post-outcome information may be used to write the checklist.
- The checklist SHA256 is recorded and rechecked during every run.

This notebook intentionally stops after cohort selection if the frozen checklist file is missing.


In [ ]:
from pathlib import Path
import os, json, hashlib

MODEL = "gemma-4-31b-it-qat-w4a16-ct"
SEED = 20261111
N_EVAL_TASKS = 12
N_PREFLIGHT_TASKS = 2
N_REPS = 2
CONDITIONS = ["O_operational", "OS_semantic"]

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
TASKS_PATH = DATA_DIR / "tasks.jsonl"
SNAPSHOTS_DIR = DATA_DIR / "snapshots"
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other") / MODEL / "2"

STUDY_ROOT = WORKING_DIR / "Gemma4_Operational_vs_Semantic_12x2x2"
CHECKPOINT_ZIP = WORKING_DIR / "Gemma4_Operational_vs_Semantic_12x2x2_Checkpoint.zip"
FINAL_ZIP = WORKING_DIR / "Gemma4_Operational_vs_Semantic_12x2x2_Research.zip"

# Filled in manually after the notebook exports the cohort/template.
SPEC_FILENAME = "semantic_specs_frozen.csv"
SPEC_CANDIDATES = [
    WORKING_DIR / SPEC_FILENAME,
]
# Also search attached Kaggle inputs recursively below.

PRIOR_TASK_IDS = set(['fastapi_11194', 'fastapi_11355', 'fastapi_13207', 'fastapi_13537', 'fastapi_13713', 'fastapi_13786', 'fastapi_14262', 'fastapi_14266', 'fastapi_14303', 'fastapi_14306', 'fastapi_14349', 'fastapi_14361', 'fastapi_14372', 'fastapi_14430', 'fastapi_14448', 'fastapi_14455', 'fastapi_14479', 'fastapi_14482', 'fastapi_14485', 'fastapi_14487', 'fastapi_14583', 'fastapi_14605', 'fastapi_14616', 'fastapi_14791', 'fastapi_14794', 'fastapi_14851', 'fastapi_14962', 'fastapi_14964', 'fastapi_14986', 'fastapi_15023', 'fastapi_15030', 'fastapi_15280', 'fastapi_15588', 'fastapi_15589', 'fastapi_15661', 'fastapi_15745', 'fastapi_15785', 'fastapi_5077', 'fastapi_9425', 'fastapi_9555', 'httpx_3672', 'requests_6589', 'requests_6592', 'requests_6629', 'requests_6644', 'requests_6757', 'requests_7205', 'requests_7309', 'requests_7315', 'requests_7328', 'requests_7427', 'requests_7433', 'requests_7502', 'requests_7505', 'rich_2725', 'rich_2943', 'rich_3006', 'rich_3043', 'rich_3052', 'rich_3061', 'rich_3063', 'rich_3064', 'rich_3067', 'rich_3105', 'rich_3130', 'rich_3180', 'rich_3278', 'rich_3296', 'rich_3454', 'rich_3469', 'rich_3472', 'rich_3480', 'rich_3486', 'rich_3518', 'rich_3521', 'rich_3535', 'rich_3676', 'rich_3718', 'rich_3782', 'rich_3882', 'rich_3894', 'rich_3905', 'rich_3930', 'rich_3934', 'rich_3938', 'rich_3942', 'rich_3944', 'rich_4070', 'rich_4075', 'rich_4076', 'rich_4077'])
PRIOR_REPOS = set(['fastapi', 'httpx', 'requests', 'rich'])

STUDY_ROOT.mkdir(parents=True, exist_ok=True)
print("Previously used task IDs excluded:", len(PRIOR_TASK_IDS))
print("Previously represented repo prefixes:", sorted(PRIOR_REPOS))


## 1. Install the official competition wheelhouse

In [ ]:
# Run once per runtime. Adapted from the ALT-v1 baseline notebook.
# It installs the competition's offline wheel files.

import glob
import importlib
import logging
import os
import subprocess
import sys
from pathlib import Path

# ── 1. Environment variables ─────────────────────────────────────────────
# Set these before importing transformers / vLLM / LiteLLM, because those
# libraries read them once at import time.
os.environ.update({
    # transformers: PyTorch only (skip TF/Flax), never try to download models
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
    "TRANSFORMERS_OFFLINE": "1",
    # vLLM: no telemetry, spawn worker processes (safe with CUDA in notebooks),
    # allow up to 20 min for the engine to load the model, and count CUDA graphs
    # when estimating GPU memory
    "VLLM_NO_USAGE_STATS": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    # Multi-GPU communication: only log warnings
    "NCCL_DEBUG": "WARN",
    # LiteLLM: use the bundled cost map (no internet fetch), no telemetry, less logging
    "LITELLM_LOCAL_MODEL_COST_MAP": "True",
    "LITELLM_TELEMETRY": "False",
    "LITELLM_LOG": "WARNING",
    # PyTorch: lower memory fragmentation on the GPU
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # OpenTelemetry: off (it would try to export traces over the network)
    "OTEL_SDK_DISABLED": "true",
    "OTEL_PYTHON_DISABLED": "true",
})
# The OpenAI-compatible clients require some API key. The local vLLM server
# doesn't check it, so "EMPTY" is enough. setdefault keeps any real key you set.
os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


# ── 2. Quiet noisy loggers ───────────────────────────────────────────────
NOISY_LOGGERS = [
    "LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm",
    "httpx", "httpcore", "openai", "urllib3",
    "opentelemetry", "opentelemetry.context",
]

def configure_logging() -> None:
    for name in NOISY_LOGGERS:
        logger = logging.getLogger(name)
        logger.setLevel(logging.WARNING)       # hide INFO/DEBUG messages
        if "litellm" in name.lower():
            logger.propagate = False           # stop LiteLLM messages appearing twice
            logger.handlers.clear()            # drop LiteLLM's own handlers

configure_logging()


# ── 3. Remove broken cutlass .pth hooks ──────────────────────────────────
# Python runs .pth files at startup. The cutlass ones in Kaggle's image can
# crash imports, so delete them if they're there.
for pth in glob.glob("/usr/local/lib/python*/*-packages/*cutlass*.pth"):
    Path(pth).unlink(missing_ok=True)


# ── 4. Restore '+cu128' in wheel filenames ───────────────────────────────
# Kaggle datasets drop the '+' from names like 'torch-2.x+cu128-...whl', and pip
# rejects a wheel whose filename doesn't match its metadata. So create correctly
# named links in /tmp. /kaggle/input is read-only, so we can't rename in place.
# Wheelhouse location varies by Kaggle attachment style; require at least one wheel.

# Robust wheelhouse discovery: Kaggle input mount paths can differ by slug/version.
WHEELHOUSE_HINTS = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),
]

def _find_wheelhouse():
    # Prefer known locations first.
    for p in WHEELHOUSE_HINTS:
        if p.exists() and any(p.rglob("*.whl")):
            return p

    # Fall back to searching all attached Kaggle inputs for a directory
    # that actually contains wheel files.
    root = Path("/kaggle/input")
    if root.exists():
        dirs = {}
        for whl in root.rglob("*.whl"):
            dirs.setdefault(whl.parent, 0)
            dirs[whl.parent] += 1
        if dirs:
            # Pick the directory containing the largest number of wheels.
            return max(dirs, key=dirs.get)
    return None

WHEELHOUSE_DIR = _find_wheelhouse()

if WHEELHOUSE_DIR is None:
    attached = []
    root = Path("/kaggle/input")
    if root.exists():
        attached = sorted(str(p) for p in root.iterdir())
    raise AssertionError(
        "Gemma Developer Agent wheelhouse not found. "
        "Attach the competition wheelhouse dataset to this Kaggle notebook, then rerun. "
        f"Currently attached top-level inputs: {attached}"
    )

print("Using wheelhouse:", WHEELHOUSE_DIR)
FIXED_DIR = Path("/tmp/wheelhouse")
FIXED_DIR.mkdir(parents=True, exist_ok=True)

for whl in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in whl.name.lower():          # skip the cutlass wheel (same breakage as step 3)
        continue
    name = whl.name
    if "cu128" in name and "+" not in name:    # put the missing '+' back
        name = name.replace("cu128", "+cu128")
    link = FIXED_DIR / name
    if link.is_symlink() and not link.exists():
        link.unlink()
    if not link.exists():
        link.symlink_to(whl)                   # a symlink is instant; no need to copy GBs


# ── 5. Install offline ───────────────────────────────────────────────────
# --no-deps: the wheelhouse already has every dependency, so pip never needs the internet
# --force-reinstall: replace the versions preinstalled in Kaggle's image
wheels = sorted(str(w) for w in FIXED_DIR.glob("*.whl"))
assert wheels, f"No wheels found: {WHEELHOUSE_DIR}"
print(f"Installing competition wheelhouse ({len(wheels)} wheels)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels],
    check=True,                                # stop the notebook if the install fails
)

importlib.invalidate_caches()                  # make this process see the new packages
configure_logging()                            # newly installed packages may reset log levels
print("Wheelhouse installation complete.")

## 2. Select and freeze 14 fresh issues, then require a frozen semantic checklist

In [ ]:
import collections, random, hashlib, math, copy
import pandas as pd
from swegemma.models import load_tasks
from IPython.display import display, FileLink

assert TASKS_PATH.is_file(), f"Missing tasks: {TASKS_PATH}"
assert SNAPSHOTS_DIR.is_dir(), f"Missing snapshots: {SNAPSHOTS_DIR}"

tasks = load_tasks(TASKS_PATH)
tasks_by_id = {t.instance_id: t for t in tasks}

def repo_prefix(task):
    tid = str(task.instance_id)
    return tid.split("_", 1)[0].lower()

def stable_key(tid):
    return hashlib.sha256(f"{SEED}|{tid}".encode()).hexdigest()

eligible = []
for t in tasks:
    tid = str(t.instance_id)
    if tid in PRIOR_TASK_IDS:
        continue
    # Require a local snapshot directory/file reference to exist in the official input.
    # The evaluator will enforce the exact snapshot later.
    wc = len(str(t.problem_statement).split())
    if wc < 8 or wc > 1200:
        continue
    eligible.append(t)

assert len(eligible) >= N_EVAL_TASKS + N_PREFLIGHT_TASKS, (
    f"Only {len(eligible)} fresh eligible tasks after exclusions."
)

# Prefer repository prefixes that did not appear in the preserved corpus.
new_repo = [t for t in eligible if repo_prefix(t) not in PRIOR_REPOS]
old_repo = [t for t in eligible if repo_prefix(t) in PRIOR_REPOS]

# Diversity-aware deterministic selection: round-robin over repos, with a max of 3
# tasks from any one repo unless necessary to fill the cohort.
def diverse_pick(pool, n, already=None):
    already = list(already or [])
    chosen = []
    counts = collections.Counter(repo_prefix(t) for t in already)
    grouped = collections.defaultdict(list)
    for t in pool:
        grouped[repo_prefix(t)].append(t)
    for r in grouped:
        grouped[r] = sorted(grouped[r], key=lambda t: stable_key(t.instance_id))
    repo_order = sorted(grouped, key=lambda r: hashlib.sha256(f"{SEED}|repo|{r}".encode()).hexdigest())

    while len(chosen) < n:
        progressed = False
        for r in repo_order:
            if len(chosen) >= n:
                break
            if not grouped[r]:
                continue
            if counts[r] >= 3:
                continue
            t = grouped[r].pop(0)
            chosen.append(t); counts[r] += 1; progressed = True
        if not progressed:
            break
    # Relax repo cap if needed.
    leftovers = []
    for r, vals in grouped.items():
        leftovers.extend(vals)
    leftovers = sorted(leftovers, key=lambda t: stable_key(t.instance_id))
    for t in leftovers:
        if len(chosen) >= n:
            break
        chosen.append(t)
    return chosen

need = N_EVAL_TASKS + N_PREFLIGHT_TASKS
selected_tasks = diverse_pick(new_repo, min(need, len(new_repo)))
if len(selected_tasks) < need:
    used_ids = {t.instance_id for t in selected_tasks}
    fill_pool = [t for t in old_repo if t.instance_id not in used_ids]
    selected_tasks += diverse_pick(fill_pool, need-len(selected_tasks), already=selected_tasks)

assert len(selected_tasks) == need
selected_ids = [str(t.instance_id) for t in selected_tasks]
assert len(set(selected_ids)) == need
assert not (set(selected_ids) & PRIOR_TASK_IDS)

preflight_ids = selected_ids[:N_PREFLIGHT_TASKS]
eval_ids = selected_ids[N_PREFLIGHT_TASKS:]
assert len(eval_ids) == N_EVAL_TASKS

records = []
for t in selected_tasks:
    tid = str(t.instance_id)
    role = "preflight" if tid in preflight_ids else "evaluation"
    records.append({
        "task_id": tid,
        "role": role,
        "repo": str(t.repo),
        "repo_prefix": repo_prefix(t),
        "base_commit": str(t.base_commit),
        "word_count": len(str(t.problem_statement).split()),
        "problem_statement": str(t.problem_statement),  # PRIVATE working aid; do not publish.
    })

cohort_df = pd.DataFrame(records)
cohort_df.to_csv(STUDY_ROOT/"cohort_PRIVATE.csv", index=False)
cohort_df.drop(columns=["problem_statement"]).to_csv(STUDY_ROOT/"cohort_public_metadata.csv", index=False)

cohort = {
    "seed": SEED,
    "selection_rule": "fresh IDs excluding preserved 416 corpus; prefer unseen repo prefixes; deterministic diversity-aware selection",
    "n_preflight": N_PREFLIGHT_TASKS,
    "n_evaluation": N_EVAL_TASKS,
    "preflight_ids": preflight_ids,
    "evaluation_ids": eval_ids,
    "task_catalog_sha256": hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest(),
}
(STUDY_ROOT/"cohort.json").write_text(json.dumps(cohort, indent=2)+"\n")

display(cohort_df[["task_id","role","repo_prefix","repo","word_count"]])

# Build a PRIVATE checklist template with the issue statement included for authoring convenience.
template_path = WORKING_DIR/"semantic_specs_template_PRIVATE.csv"
template = cohort_df[["task_id","role","repo","problem_statement"]].copy()
for col in [
    "neutral_restatement",
    "required_behavior",
    "edge_cases",
    "preserved_invariants",
    "source_notes",
    "spec_author",
    "spec_reviewer",
    "review_status",
]:
    template[col] = ""
template.to_csv(template_path, index=False)

# Find a filled frozen spec either in /kaggle/working or attached Kaggle inputs.
spec_path = next((p for p in SPEC_CANDIDATES if p.exists()), None)
if spec_path is None:
    input_matches = list(Path("/kaggle/input").rglob(SPEC_FILENAME))
    if input_matches:
        spec_path = input_matches[0]

if spec_path is None:
    print("\nSTOP: semantic specifications must be authored and frozen before any model run.")
    print("1. Download this PRIVATE template:")
    display(FileLink(str(template_path)))
    print("2. Fill every row. Use only the issue statement and repository-visible docs/tests/code.")
    print("3. Do NOT use reference patches, hidden tests, evaluator outputs, or candidate-run outcomes.")
    print("4. Save as semantic_specs_frozen.csv and upload it to /kaggle/working or attach it as an input.")
    print("5. Rerun this cell. The notebook will record its SHA256 and refuse mid-study changes.")
    raise RuntimeError("Frozen semantic specification file required before inference.")

spec_df = pd.read_csv(spec_path).fillna("")
assert set(spec_df["task_id"].astype(str)) == set(selected_ids), (
    "Frozen spec task IDs must exactly match the selected 14-task cohort."
)
required_cols = [
    "task_id","neutral_restatement","required_behavior","edge_cases",
    "preserved_invariants","source_notes","spec_author","review_status"
]
missing_cols = [c for c in required_cols if c not in spec_df.columns]
assert not missing_cols, f"Missing spec columns: {missing_cols}"
for c in required_cols[1:]:
    empty = spec_df[spec_df[c].astype(str).str.strip()==""]["task_id"].tolist()
    assert not empty, f"Column {c!r} is empty for: {empty}"

allowed_status = {"approved","self-reviewed"}
bad = spec_df[~spec_df["review_status"].str.lower().isin(allowed_status)]
assert bad.empty, "review_status must be 'approved' or 'self-reviewed'."

SPEC_PATH = Path(spec_path)
SPEC_SHA256 = hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest()
shutil_target = STUDY_ROOT/"semantic_specs_frozen_PRIVATE.csv"
shutil_target.write_bytes(SPEC_PATH.read_bytes())

# Freeze a public-safe manifest without issue text or semantic content.
spec_manifest = {
    "spec_filename": SPEC_FILENAME,
    "sha256": SPEC_SHA256,
    "n_rows": len(spec_df),
    "task_ids": sorted(spec_df.task_id.astype(str)),
    "authoring_rule": "pre-outcome; no reference patch/hidden test/evaluator-output use",
    "review_status_counts": spec_df["review_status"].str.lower().value_counts().to_dict(),
}
(STUDY_ROOT/"semantic_spec_manifest.json").write_text(json.dumps(spec_manifest, indent=2)+"\n")

spec_by_id = spec_df.set_index("task_id").to_dict("index")
selected = selected_ids
print("\nFrozen specification accepted.")
print("SHA256:", SPEC_SHA256)
print("Preflight:", preflight_ids)
print("Evaluation:", eval_ids)


## 3. Build one operationally supported agent shared by both conditions

In [ ]:
import base64, io, zipfile, difflib, shutil, yaml
from adk_submission import validate_directory
from adk_submission.discovery import discover_declared_models
from swegemma.config import build_submission_limits

ORIGINAL_ALT_V1_SHA256 = '277b20147e89904e1da3386515cef9bc3ab662c7775441d851a0a66405652e08'
ORIGINAL_ALT_V1_B64 = 'UEsDBBQAAAAIAAAAIQAQu8lwwwAAADoBAAAKAAAAYWdlbnQueWFtbFWQQW4DIQxF95yCHoBUUaMu5jLIAw5jCRsKRqPcPnQSJe3KevbXt78FGBfbd/SQUNRwiZgXm5AZ3MV9nVdH6n5A3X6B87cLaki6thGUiiz2gyTkEdHWVrhq/+y3rsgnjkZLyX0x1jrbhvhQpqXEByNEf6WMB2EkfdPeSPGNCdV3BR39wD5WnuoKGrbXPMyjvSClbS3tqUNoYfOdmDK0Q/FfPo1Sg7qZGRsbzJWhiM4X/NYrpT/RHo0ZDbhmknS6AWdzB1BLAwQUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2MUQrCMBAF/3OKPYFUUTG5zFLbNQm1uyG+hR7f+OPfMAwD2Zv0Gd4l0XS6BFjj9sN4C/t8sDmag2Gb6CfR4xxHU6puVTMvpq+aUyD6q6evWZDoOsX78FWXt6/CKOa5YBzQXcIXUEsDBBQAAAAIAAAAIQCBs8SDSwAAAF4AAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIwMjEACucmVsSX5OfnxCcn5uSARC1ggkD18bmZeaUlqUBhuGhpUR6Ia8AFAFBLAwQUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAHByb21wdHMvc3lzdGVtLm1kjVnbbhtHEn33VzQSICsHJH1JAmRlYAGtLCPCKrZhx7vIE9mcaZK9nJkeT/eIYr5+z6nquUhBgH2xNeT0papOnTpV/D30xnbO2MbYPoUm1KGPJoZdOvFj1+x941y3Mr+HvjP/DdtLU9ujM+ngTKxtVbmYTBG6zhX4/2CbPb4L8vXnD18+Xd/gy9IZ35gXp9AdY2sLLOQLNslbB1+WrjEJ+0SzC5186GPsndm6KpxMa2Ncmd+evnpyuN6p8ynho10X6tnC5B7SQk+BCQ8WV2ts7eLC4J9o9/wrnVsXYXaJYw723sO8yGtOu9B+MbN1hd/5wiYfmtWzZ9+aX64+vTV3t7/e/vb52dK8/hkWh8oU8Ibu+LOpfdPjnrJhSLZamf8ccE/nsWFnur6JJvS4I/enX3yzN6lzPNrsO1u60lisTnhemSssLH1Sn7kHT/OL0DcJZrh7bOt35ow4Nnjo5BpmE/tt7dO6tak4bGSLvuEm2FjOsdXJnhFohA7X/MN1YQVT3usOnbPJwVmh9LuzQUxKV7nkxPNm5xFzc7Hhw/r7VXveLMzm+zUf9cE253SgQX1TYjd5Mb7gW+1Z3vKN51MRmt2w6jkjDNt9HJBYw92F2Vlf9Z3j3T4XHY3J5++DYCrVrQlNdaaFZ/nKVG6X8tEzyG1dERB8oKlLJuzE8dnR4qPJegTHWLPr4UWxN/bwmrnYEg7ZgskWs+LffUMY4qn0cCg2oT1X6mECD1ecOW8hGLFlaTbL48acCAyGr7DNdIsqhCMhEj2yZ7KD21YIR+tgYFN4QpjZWyFk5RkuiYk5WU4b9RHXjs52xWEdfe0r262Zkrz33iX5e904vz9sQxc3jPb0OVAEJ7VEEBLwbDqX+q4xTcgBjg5+kpS4vrq7Mx/vrt7j4GtJhFfLny9hBtLGicG84Sp/9/flq9eXAmohFvPrl8+/GWShMxmjFtnbd4ibhHR7VlS/er0yt4p1Wo0sAjYWSkhn8tMWPl7ueyS5ZkzO5zrA9ZU/uups2opwaMIJrgR/BJghr8L7JiaPU7ZuIDT40XzEEXJ3GoyXYLtmzWqy9Ifl65eXBt5mutDWvH54xbx+CQiAUMtLnIw8PSGN2yp0Sinm+uCK4wK2PoC1JHG5MgNoh3SJB3MCdSAyniDb7TagygrsCFfBOU+ynfH4+OnD9c3bL59u8PCtebUyNw+pIxH+idHMBa40EtjzZ+8ecekCfiNuAQt6OLpqdzmjVcXzrm8KbrUwRQW2Joau724F+m9mL7uuwzeZgsVP7qFwrVyCdDy824rrB7Dd2wpkjIXIhrZP+pJv8GfMVaTz+73TwrHt92/Iid0ZUcUhhQX8a9fwDGx58R7/gTTrNp0XiMMeHrjHB0hg4n1hyr6t6BdWiNLtbF+lSG4iI+Z9QVsgSIA64mBggnZIkj8uHkhMuC65zlZ6zbqHH23bIhXVHxXzVXHLw1cSKSD8TnIGAPgSB88hsIf4tMqVnhgj90lt0RzPhNC4gm7uYCV4b4sajrMG2kjTu48K7t9iPi6e622oopQnrxXFdTFnceXuLbYQOtN7+QYVnxQH6gRv1DVeXMntCdd951pFL/DtymV2G5bCOY1eYXCU5+2AzOE0rq/iUilf7goMHxFIoQU5HeG2qAqoYoiViYfQVyX/O6F6xuk92a6iH5q+3gItoF0SZyvx3laSzv+UAsN6MnnodAhxxNuca4/gkEb4iA5sIDnyokzPSAMFxM53cJVecEGKJr4VE/nWWwfbXF4umNpbj73vwNdJ7ctbw9F2i0Qgo9S8Ld+mWeqvo4OvRXDkbKEnOuSMSoOyR7mEQZXLVTRHHxEK0acAdFMc0BUfmkLVT7IdygELs3J5uRA2zOsyGHzdVo5ppozSVn1UQAwvKITkPBGMSMsM2y1qScbRGKKOMlK9LV6ejoxZdeKDPU4SgfUEeahKdEoESgR1ZFpfEEjT1gvehJWR+64FJEQXqg6WVi7vjgrha/1CfP7jy+zqljJBXtFSnrTCpAOV1XAzvjQmw21TVH1JPR36PaLZgwqhUaSmUB8jriJZUL6xYfT7xoL85iykfCkMCpVd4iJ7xumdXBu0d8lCUWZYi6yU2jaKW0aw7ULZFyyv4iZ970loHt9EXgiNlo4MU5bLR6fvnFw230CWVEiamMWqRMNveWNzQReLCJkVjdDOioeEpgwFqYioep4LanuWhM60wTyiR7knHQ5SydALYCPo2h7JAg5er2Hdej2cIcWMTvVRaiXQva1y7gDDSJVIsz45OmcCpThGOogx+welR4nwNhhRBcx6vB9E448VldkFB0laj2nM5B+2sk1kM6Oe+wrxon3GcqAWZXu1lzdQxMTGo5aMPJJDG2UFHmBWvRClVNuzaFpkQGBSS1JAvR7gjmVhLpg+vOdi6B9E/TOsnt4HHqJzOd3BKwJBrVU/QFUAC0Olwp9TKmkrGLl8BIFUOXPNUIaqXOdCMNTCsbqp8ptH+HRAfRD9q9q5OASSslV2Y/WG25QPte6wL3R6GWFzTe69a3rkL56pwOYCc+xG1ded+9qjuMYchMkyEiUwQhFXa62AnkF2egBVP2HKjvyYq6keNCMMyAdQrQTC5i66G6zIcV0N3YiAh02R1kZESpoiwEXl8xV6KKDtFKZ6kJsm3IQwDlL7hfw+atRj0flW6ErIaM6d+U5f+0AFRi5BKr5BJkXXpclNCJ802SYUBfvmIYYCsEwTuDmkNu/jk1jNoEife1CvQ1vjkq67p5JHpWBxEbKhQuGCUvOqcxLOcZ0ipXY16pWQkH/IRAZzF7JEYHSGygv1PD0Lb6ulSMIcHRLME1klml3nGAKd019Aw/yLtXZGcH+mbAXrFoqF0p/FAJTitx5q5wyTK7Yp086RYl+44+RZHX9VgqIl6UyVDQIa9KYk4XKoA4CGD3Uc2tqnxWVGJZE6dSoJBysNMfFXPanh6PDjuSm0vslf97bzVmYO15UfQnTFr/QZrQsRC3zSYhVAItggeJg0MnLhnUibVbClZMoS5VlSCoV8s16j10nrNScC0ohm+t48X4hSO2ucxymTCnICXE1HsH+Zz4jCjsMh9+C6wrMPCFIarUiGVaatWXsyNiPaxWiDwm+GlkUg/wQuj1laIAOWRoZtMxlLrGjqUHDRzEtsJKy8ee6+e6XDeZMkDPQLw+z+1NWQ21nOkDSerSvQ+GqoyUjafgzlUhWGjUcElUw4AP+djenq4y2cm8dL9IQreimKWmXyCAf7xXXsihcbZf4fV+bf2u1eDOLntc6/nktCZhkwjjxy96r6QEKzlJbpm++P7kx/cI70zUYajkaq1Sb52lF0vfrp5VitapOnLjpSYoF+IWOoByw3ywez/GqWR5P33Ggt2OjqH7h8HNTIBitdu5HyMH6F4vlfCHRx/bvMpMp8nCBl8kG0hWn9Q8754ZPSJisSRuWEhjPOCA6w1ZZPUUyN20oaKJEAT/tGcUo4j0XA+m4AtWp219x7KDxGOSe4VX//xUEsCvPaREnnmKkyd5gVXiHxRa5TDAWfBeacE6h6lxgtSNBMvs2oJjY4zRVHyd150FhfWuBgM/REecwjZUAQ9dOKsAF56AaigmWoQTlGPLAVhCZN6CyWS637331npnlIbgQIwtV8TmIrfnzwlE59M+8f4Y0tAPZ4eDLpLA1AHjdRAtlmtkGcTSONqAv1czmf83WoUahuVF4MYkwMLJ6mWY5YS5gvl5uxDZT7KBiuQ4P2sRb8XebOdNItcTh0mHkwqRaPJiycqyhmlSHyBGIs2W+mYd4I0ZOlboOC5sZJ+2TOFILMVoN0kg4tB4ArYhvGUX/r/DWOQ6+nM+jBlo18sI7+D2f+YV5u5J4bWbsenMjPxfl5Yo7m5aXKWZkSsp4Pk8I8bVYtysHeDDy5kcaNbiZxoz3eMjoqDf4a0UP7sI7sssR4kAna25vr28+3H96bT1/ubjjnv6q3ft/z5xHh+stBhs4mjDrFGKaFYV4ZpHZINS0oxA21W1YmYOZR98e58pFhrsy+c2VFm85OymkvztcvTeuRLyLKgPP/q/1TjaSvyk8MQExHWRy2IsdKPVI6FR3Cwc8ZJpcm5ZLL9MKyJo3jxDwRyywuNWWcypI0Sl/qMPh8sqLcPqeeuS4K1g6yFv7PvzHZ6YzFKJMVIaPvsj4exY2I4umXjJD7buHvAemNc6V0JsNeN5I8rJCjvsPhDUqnTAqHlMs/FPwPUEsBAhQDFAAAAAgAAAAhABC7yXDDAAAAOgEAAAoAAAAAAAAAAAAAAKSBAAAAAGFnZW50LnlhbWxQSwECFAMUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAAAAAAAAAAAApIHrAAAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAAAAhAIGzxINLAAAAXgAAABAAAAAAAAAAAAAAAKSBgwEAAGV2YWxfY29uZmlnLnlhbWxQSwECFAMUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAAAAAAAAAAAApIH8AQAAcHJvbXB0cy9zeXN0ZW0ubWRQSwUGAAAAAAQABAD4AAAAhQ4AAAAA'

raw = base64.b64decode(ORIGINAL_ALT_V1_B64)
assert hashlib.sha256(raw).hexdigest() == ORIGINAL_ALT_V1_SHA256
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None
    baseline_files = {n:z.read(n) for n in z.namelist()}

assert set(baseline_files) == {
    "agent.yaml", "eval_config.yaml", "configs/sampling.yaml", "prompts/system.md"
}
base_prompt = baseline_files["prompts/system.md"].decode()

# IDENTICAL agent/system prompt for O and O+S.
# The treatment difference is injected only into the task problem statement.
operational_support = r"""

OPERATIONAL SUPPORT FOR THIS CONTROLLED STUDY
- Start with one targeted reproduction or existing relevant test before broad exploration.
- If an edit operation fails, re-read the exact surrounding file region and retry once with the smallest viable edit.
- When direct Python verification could import an installed package instead of the edited workspace,
  force the workspace path (for example PYTHONPATH=src where appropriate) and inspect __file__ when useful.
- Before submitting a non-empty patch, run a focused behavior check. The task supplement tells you which marker
  to prefix to that check. Do not edit tests merely to make verification pass.
- Preserve unrelated behavior and avoid broad refactors unless the issue requires them.
"""

shared_prompt = base_prompt + operational_support
PROMPTS = {"shared_operational_agent": shared_prompt}

AGENT_ROOT = STUDY_ROOT / "agents"
AGENT_DIRS = {}
limits, generation_constraints = build_submission_limits()

for label, prompt in PROMPTS.items():
    folder = AGENT_ROOT / label
    if folder.exists():
        shutil.rmtree(folder)
    for name, blob in baseline_files.items():
        target = folder / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(prompt.encode() if name == "prompts/system.md" else blob)
    validate_directory(folder, limits=limits)
    assert discover_declared_models(folder) == {MODEL}
    AGENT_DIRS[label] = folder

study_plan = {
    "study": "Gemma4_Operational_vs_Semantic_12x2x2",
    "primary_comparison": "OS_semantic minus O_operational on paired evaluator resolution",
    "n_eval_tasks": N_EVAL_TASKS,
    "n_repetitions": N_REPS,
    "n_eval_runs": N_EVAL_TASKS * 2 * N_REPS,
    "n_preflight_runs": N_PREFLIGHT_TASKS * 2,
    "conditions": {
        "O_operational": "neutral restatement + generic review guidance",
        "OS_semantic": "same neutral restatement + frozen issue-specific behavioral checklist",
    },
    "shared_agent_config": {
        "model": MODEL, "temperature": 0.2, "thinking_budget": 4096,
        "max_tool_calls": 28, "max_turns": 80, "max_time_minutes": 8,
        "declared_tools": 9,
    },
    "baseline_archive_sha256": ORIGINAL_ALT_V1_SHA256,
    "semantic_spec_sha256": SPEC_SHA256,
}
(STUDY_ROOT/"experiment_plan.json").write_text(json.dumps(study_plan, indent=2)+"\n")

print("Shared operational agent prepared for both conditions.")
print("Frozen semantic spec SHA256:", SPEC_SHA256)


## 4. Start Gemma 4 31B once

In [ ]:
import os, requests, torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

assert torch.cuda.device_count()==4, 'Choose Kaggle 4 × L4 GPUs'
assert MODEL_PATH.is_dir(), f'Attach model version 2: {MODEL_PATH}'
litellm.drop_params=True
adapters=discover_adapters(str(AGENT_DIRS['shared_operational_agent']),adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert not (getattr(adapters,'adapters',None) or {}), 'Original agent should not contain adapters'

# Reuse the server if just running the next batch within the same Kaggle session.
ready=False
signature=(str(MODEL_PATH),4,32768,0.80)
if globals().get('server_instance') is not None and globals().get('SERVER_SIGNATURE')==signature:
    try: ready=requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=5).ok
    except Exception: pass
if not ready:
    if globals().get('server_instance') is not None:
        try: server_instance.stop()
        except Exception as exc: print('Old server stop warning:',exc)
    config=VllmConfig(
        model=str(MODEL_PATH),host='127.0.0.1',port=8000,
        tool_call_parser='gemma4',reasoning_parser='gemma4',enable_auto_tool_choice=True,
        max_model_len=32768,dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
        gpu_memory_utilization=0.80,enable_lora=False,tensor_parallel_size=4,
        scheduler_reserve_full_isl=False,startup_timeout=20*60)
    server_instance=VllmServer(config,adapter_manifest=adapters)
    server_instance.start()
    SERVER_SIGNATURE=signature
models=server_instance.create_model_registry(aliases=[MODEL],model_prefix='openai/',
                                           api_key=os.environ.get('LOCAL_API_KEY','EMPTY'))
requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=15).raise_for_status()
print('Gemma inference ready:',server_instance.base_url)

## 5. Run 4 excluded preflight runs + 48 randomized evaluation runs

In [ ]:
import asyncio, concurrent.futures, contextlib, io, time, traceback, zipfile, shutil, math, copy, re
import pandas as pd
from collections import Counter
from IPython.display import display, clear_output, FileLink
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator

SHARED_AGENT_DIR = AGENT_DIRS["shared_operational_agent"]
TOTAL_EVAL_RUNS = N_EVAL_TASKS * len(CONDITIONS) * N_REPS
TOTAL_PREFLIGHT_RUNS = N_PREFLIGHT_TASKS * len(CONDITIONS)
TOTAL_RUNS = TOTAL_EVAL_RUNS + TOTAL_PREFLIGHT_RUNS
RESULTS_JSON = STUDY_ROOT/"results.json"

# Recheck frozen spec before evaluator creation.
assert hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest() == SPEC_SHA256

evaluators={}
for label in CONDITIONS:
    evaluators[label]=Evaluator(EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=SNAPSHOTS_DIR,
        submission_dir=SHARED_AGENT_DIR,
        results_dir=STUDY_ROOT/"evaluation"/label,
        wheels_dir=DATA_DIR/"wheels",
        graph_dir=str(DATA_DIR/"graphs"),
        embeddings_dir=str(DATA_DIR/"embeddings"),
        models=models,
        adapter_manifest=adapters,
        sandbox="subprocess",
        timeout_seconds=240,
        verification_timeout_seconds=300,
        max_tool_calls=28,
        max_time_minutes=8,
        max_turns=80,
        limits=limits,
        generation_constraints=generation_constraints,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2,
            token_threshold=14336, event_retention_size=5),
        verbose=False,
    ))

def task_copy_with_problem(task, problem_statement):
    # swegemma Task objects are frozen dataclasses in the Kaggle runtime.
    # Create a new immutable instance rather than mutating a deepcopy.
    try:
        from dataclasses import replace, is_dataclass
        if is_dataclass(task):
            return replace(task, problem_statement=problem_statement)
    except Exception:
        pass

    # Compatibility fallback for Pydantic-style task objects.
    if hasattr(task, "model_copy"):
        return task.model_copy(update={"problem_statement": problem_statement})

    raise TypeError(
        f"Unsupported task object type for immutable problem-statement replacement: {type(task)!r}"
    )

def generic_control_guidance():
    return """GENERIC REVIEW GUIDANCE
1. Derive the required behavior from the issue and repository-visible evidence.
2. Consider at least one boundary or alternate input before finalizing.
3. Preserve behavior outside the requested scope.
4. Run one focused behavior check before submission.
Prefix that check's shell command with: echo CONTROL_BEHAVIOR_CHECK &&"""

def semantic_guidance(spec):
    return f"""FROZEN HUMAN-STRUCTURED BEHAVIORAL SPECIFICATION
Required behavior:
{spec['required_behavior']}

Edge cases / boundaries to consider:
{spec['edge_cases']}

Preserved invariants:
{spec['preserved_invariants']}

Use this specification as behavioral guidance only. It gives no implementation or file-location instruction.
Before submission, run at least one focused check derived from this specification.
Prefix that check's shell command with: echo SPECIFICATION_DRIVEN_CHECK &&"""

def augment_problem(task, condition):
    spec = spec_by_id[str(task.instance_id)]
    common = f"""CONTROLLED-STUDY SUPPLEMENT

Neutral restatement of issue facts:
{spec['neutral_restatement']}

"""
    extra = generic_control_guidance() if condition=="O_operational" else semantic_guidance(spec)
    return str(task.problem_statement) + "\n\n" + common + extra

def async_run(evaluator, task, task_index):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(
            evaluator.evaluate_task(task=task, task_index=task_index, total_tasks=TOTAL_RUNS)
        )).result()

def parse_trace(trace_path, expected_marker):
    out={
        "trace_available":False, "trace_tool_calls":None,
        "first_successful_edit_call":None, "read_calls":None,
        "run_command_calls":None, "edit_calls":None, "write_calls":None,
        "graph_calls":None, "tool_error_count":None,
        "behavior_marker_executed":False, "behavior_marker_calls":0,
        "post_marker_edit":False,
    }
    if not trace_path.exists():
        return out
    try:
        trace=json.loads(trace_path.read_text())
        steps=trace.get("steps",[])
    except Exception:
        return out
    out["trace_available"]=True
    call_index=0; counts=Counter(); errors=0; first_edit=None
    marker_seen=False; marker_calls=0; post_marker_edit=False
    for step in steps:
        if step.get("source")!="agent":
            continue
        obs=step.get("observation") or {}
        obs_text=json.dumps(obs, default=str).lower()
        step_error=('"status": "error"' in obs_text) or ("error_type" in obs_text and '"error_type": null' not in obs_text)
        for call in step.get("tool_calls") or []:
            call_index += 1
            name=call.get("function_name","")
            counts[name]+=1
            if step_error: errors += 1
            if first_edit is None and name in {"edit_file","write_file"} and not step_error:
                first_edit=call_index
            if name=="run_command":
                args=call.get("arguments") or {}
                cmd=str(args.get("command",""))
                if expected_marker in cmd:
                    marker_seen=True; marker_calls += 1
            if marker_seen and name in {"edit_file","write_file"}:
                post_marker_edit=True
    out.update({
        "trace_tool_calls":call_index,
        "first_successful_edit_call":first_edit,
        "read_calls":counts["read_file"],
        "run_command_calls":counts["run_command"],
        "edit_calls":counts["edit_file"],
        "write_calls":counts["write_file"],
        "graph_calls":sum(counts[x] for x in ["get_code_neighbors","search_similar_code","get_code_subgraph"]),
        "tool_error_count":errors,
        "behavior_marker_executed":marker_seen,
        "behavior_marker_calls":marker_calls,
        "post_marker_edit":post_marker_edit,
    })
    return out

def checkpoint_zip():
    tmp=CHECKPOINT_ZIP.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(STUDY_ROOT.rglob("*")):
            if p.is_file() and not p.is_symlink():
                z.write(p,p.relative_to(WORKING_DIR).as_posix())
    os.replace(tmp,CHECKPOINT_ZIP)

def compact_screen(results, current=None):
    clear_output(wait=True)
    df=pd.DataFrame(results)
    print(f"O vs O+S diagnostic — {len(results)}/{TOTAL_RUNS} runs complete")
    if current: print("Current:",current)
    if not df.empty:
        e=df[df["role"]=="evaluation"]
        if not e.empty:
            summary=(e.groupby("condition")
                .agg(n=("task_id","count"),
                     resolved=("resolved_local",lambda x:int(pd.Series(x).fillna(False).sum())),
                     marker=("behavior_marker_executed",lambda x:int(pd.Series(x).fillna(False).sum())),
                     median_calls=("tool_calls","median"),
                     median_s=("elapsed_seconds","median"))
                .reset_index())
            display(summary)
    print("Checkpoint:",CHECKPOINT_ZIP)

# Preflight: each condition exactly once on each preflight task.
schedule=[]
for tid in preflight_ids:
    order=sorted(CONDITIONS, key=lambda c: hashlib.sha256(f"{SEED}|preflight|{tid}|{c}".encode()).hexdigest())
    for label in order:
        schedule.append({"task_id":tid,"condition":label,"rep":0,"role":"preflight"})

# Evaluation: both conditions for each task × repetition, with deterministic randomized order.
for rep in range(1,N_REPS+1):
    task_order=sorted(eval_ids, key=lambda tid: hashlib.sha256(f"{SEED}|rep{rep}|task|{tid}".encode()).hexdigest())
    for tid in task_order:
        order=sorted(CONDITIONS, key=lambda c: hashlib.sha256(f"{SEED}|rep{rep}|{tid}|{c}".encode()).hexdigest())
        for label in order:
            schedule.append({"task_id":tid,"condition":label,"rep":rep,"role":"evaluation"})

assert len(schedule)==TOTAL_RUNS
assert len({(x["task_id"],x["condition"],x["rep"]) for x in schedule})==TOTAL_RUNS
(STUDY_ROOT/"execution_schedule.json").write_text(json.dumps(schedule,indent=2)+"\n")

results=[]
completed=set()
if RESULTS_JSON.exists():
    results=json.loads(RESULTS_JSON.read_text())
    completed={(r["task_id"],r["condition"],r["rep"]) for r in results}

artifact_root=STUDY_ROOT/"captured_output"
artifact_root.mkdir(exist_ok=True)
compact_screen(results)

for run_index,item in enumerate(schedule,1):
    tid=item["task_id"]; label=item["condition"]; rep=item["rep"]; role=item["role"]
    key=(tid,label,rep)
    if key in completed: continue

    # Refuse spec drift at every run.
    assert hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest() == SPEC_SHA256, "Frozen spec file changed mid-study."

    base_task=tasks_by_id[tid]
    augmented=augment_problem(base_task,label)
    task=task_copy_with_problem(base_task,augmented)
    supplement_words=len(augmented.split())-len(str(base_task.problem_statement).split())
    expected_marker="CONTROL_BEHAVIOR_CHECK" if label=="O_operational" else "SPECIFICATION_DRIVEN_CHECK"

    current=f"{run_index}/{TOTAL_RUNS}  {role} rep={rep}  {label}  {tid}"
    compact_screen(results,current)

    cond_art=artifact_root/label/f"rep_{rep}"
    cond_art.mkdir(parents=True,exist_ok=True)
    stdout,stderr=io.StringIO(),io.StringIO()
    started=time.time()
    status="unknown"; err=None; result_obj=None
    try:
        with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
            result_obj=async_run(evaluators[label],task,run_index-1)
        status="ok"
    except Exception as exc:
        status="exception"; err=repr(exc)
        traceback.print_exc(file=stderr)
    elapsed=time.time()-started

    # Capture evaluator result fields defensively.
    resolved=False; patch_text=""; test_exit=None; tool_calls=None
    if result_obj is not None:
        resolved=bool(getattr(result_obj,"resolved",False))
        patch_text=str(getattr(result_obj,"patch","") or "")
        test_exit=getattr(result_obj,"test_exit_code",None)
        tool_calls=getattr(result_obj,"tool_calls",None)

    # Fallback patch file from evaluator output if object field is empty.
    trace_path=STUDY_ROOT/"evaluation"/label/"traces"/f"trace_{tid}.json"
    # Evaluator may overwrite same trace filename across reps; preserve a rep-specific copy immediately.
    trace_copy=cond_art/f"{tid}.trace.json"
    if trace_path.exists():
        shutil.copy2(trace_path,trace_copy)

    tm=parse_trace(trace_copy,expected_marker)

    (cond_art/f"{tid}.stdout.txt").write_text(stdout.getvalue())
    (cond_art/f"{tid}.stderr.txt").write_text(stderr.getvalue())
    (cond_art/f"{tid}.patch").write_text(patch_text)

    row={
        "task_id":tid, "repo":str(base_task.repo), "role":role, "rep":rep,
        "condition":label, "status":status, "error":err,
        "resolved_local":resolved, "test_exit_code":test_exit,
        "tool_calls":tool_calls if tool_calls is not None else tm["trace_tool_calls"],
        "elapsed_seconds":round(elapsed,3), "patch_chars":len(patch_text),
        "supplement_words":supplement_words,
        "expected_marker":expected_marker,
        **tm,
    }
    results.append(row)
    completed.add(key)
    RESULTS_JSON.write_text(json.dumps(results,indent=2)+"\n")
    pd.DataFrame(results).to_csv(STUDY_ROOT/"results.csv",index=False)
    checkpoint_zip()

compact_screen(results)
print("All scheduled runs complete.")


## 6. Paired evaluation analysis with task-clustered uncertainty

In [ ]:
import pandas as pd, numpy as np, math
from collections import Counter

df=pd.DataFrame(results)
assert len(df)==TOTAL_RUNS, (len(df),TOTAL_RUNS)
df.to_csv(STUDY_ROOT/"results.csv",index=False)

pre=df[df.role=="preflight"].copy()
ev=df[df.role=="evaluation"].copy()
assert len(pre)==TOTAL_PREFLIGHT_RUNS
assert len(ev)==TOTAL_EVAL_RUNS

print("Preflight runs (excluded from all effect estimates):")
display(pre[["task_id","condition","status","resolved_local","tool_calls","elapsed_seconds","patch_chars"]])

summary=(ev.groupby("condition")
    .agg(
        runs=("task_id","count"),
        resolved=("resolved_local",lambda x:int(pd.Series(x).fillna(False).sum())),
        nonempty=("patch_chars",lambda x:int((pd.Series(x)>0).sum())),
        marker_uptake=("behavior_marker_executed",lambda x:int(pd.Series(x).fillna(False).sum())),
        post_marker_edit=("post_marker_edit",lambda x:int(pd.Series(x).fillna(False).sum())),
        median_calls=("tool_calls","median"),
        median_runtime_s=("elapsed_seconds","median"),
        median_patch_chars=("patch_chars","median"),
        median_supplement_words=("supplement_words","median"),
    ).reset_index())
print("\nCondition summary:")
display(summary)

# Paired row-level comparison within each task x repetition.
wide=ev.pivot_table(index=["task_id","rep"],columns="condition",values="resolved_local",aggfunc="first")
assert not wide.isna().any().any()
wide=wide.astype(int)
wide["diff"]=wide["OS_semantic"]-wide["O_operational"]
b=int(((wide["O_operational"]==1)&(wide["OS_semantic"]==0)).sum())
c=int(((wide["O_operational"]==0)&(wide["OS_semantic"]==1)).sum())
print("\nPaired task-repetition discordance:")
print("O-only:",b," O+S-only:",c," paired mean difference:",wide["diff"].mean())

# Exact two-sided McNemar / sign-test p on discordant pairs.
n_disc=b+c
if n_disc:
    from math import comb
    k=min(b,c)
    p_exact=min(1.0, 2*sum(comb(n_disc,i) for i in range(k+1))/(2**n_disc))
else:
    p_exact=1.0
print("Exact paired p:",p_exact)

# Cluster bootstrap at the TASK level: keep both repetitions together.
task_effect=wide.groupby(level="task_id")["diff"].mean()
rng=np.random.default_rng(20261111)
boot=[]
vals=task_effect.to_numpy()
for _ in range(20000):
    sample=rng.choice(vals,size=len(vals),replace=True)
    boot.append(sample.mean())
ci=np.quantile(boot,[0.025,0.975])
effect=float(vals.mean())
print(f"Task-clustered mean resolution difference O+S - O: {effect:+.3f}")
print(f"Task-cluster bootstrap 95% CI: [{ci[0]:+.3f}, {ci[1]:+.3f}]")

# Per-task transparency table.
task_table=wide.reset_index().pivot(index="task_id",columns="rep",values=["O_operational","OS_semantic","diff"])
task_table.to_csv(STUDY_ROOT/"paired_task_outcomes.csv")
display(task_table)

analysis={
    "n_unique_eval_tasks":int(ev.task_id.nunique()),
    "n_eval_runs":len(ev),
    "O_resolved":int(ev[ev.condition=="O_operational"].resolved_local.sum()),
    "OS_resolved":int(ev[ev.condition=="OS_semantic"].resolved_local.sum()),
    "paired_O_only":b,
    "paired_OS_only":c,
    "exact_p":p_exact,
    "task_clustered_effect":effect,
    "task_clustered_bootstrap_ci95":[float(ci[0]),float(ci[1])],
    "spec_sha256":SPEC_SHA256,
}
(STUDY_ROOT/"analysis_summary.json").write_text(json.dumps(analysis,indent=2)+"\n")
print("\nAnalysis summary:")
print(json.dumps(analysis,indent=2))


## 7. Extended post-run analysis

This section turns the experiment into a paper-ready diagnostic study rather than stopping at the primary resolution comparison.

It reports paired outcomes, solved-frontier expansion, treatment uptake, resource use, patch-production sensitivity, replication transitions, evaluator-status categories, and a manual semantic-quality audit template.

**Important:** marker execution is only a treatment-uptake proxy. It does not prove that the supplied specification was used meaningfully.


In [ ]:
import pandas as pd, numpy as np, math, json

ev = pd.DataFrame(results)
ev = ev[ev["role"]=="evaluation"].copy()
assert len(ev)==TOTAL_EVAL_RUNS
ev["resolved_local"] = ev["resolved_local"].fillna(False).astype(bool)
ev["nonempty_patch"] = ev["patch_chars"].fillna(0).astype(float) > 0

pair = ev.pivot_table(index=["task_id","rep"], columns="condition", values="resolved_local", aggfunc="first")
pair = pair[["O_operational","OS_semantic"]].astype(int)
pair["diff_OS_minus_O"] = pair["OS_semantic"] - pair["O_operational"]

o_only = int(((pair.O_operational==1)&(pair.OS_semantic==0)).sum())
os_only = int(((pair.O_operational==0)&(pair.OS_semantic==1)).sum())
discordant = o_only + os_only

if discordant:
    from math import comb
    k=min(o_only,os_only)
    p_exact=min(1.0, 2*sum(comb(discordant,i) for i in range(k+1))/(2**discordant))
else:
    p_exact=1.0

primary = {
    "eval_runs": len(ev),
    "unique_tasks": int(ev.task_id.nunique()),
    "O_resolved": int(ev.loc[ev.condition=="O_operational","resolved_local"].sum()),
    "OS_resolved": int(ev.loc[ev.condition=="OS_semantic","resolved_local"].sum()),
    "O_only_pairs": o_only,
    "OS_only_pairs": os_only,
    "discordant_pairs": discordant,
    "exact_two_sided_p": p_exact,
    "mean_paired_diff": float(pair["diff_OS_minus_O"].mean()),
}
print("PRIMARY PAIRED OUTCOME")
print(json.dumps(primary, indent=2))


### 7.2 Task-level solved frontier

This asks whether O+S solves any task that O never solves across the two repetitions. This is descriptive and does not establish an intrinsic capability boundary.


In [ ]:
frontier = (
    ev.groupby(["task_id","condition"])["resolved_local"]
      .max()
      .unstack("condition")
      .fillna(False)
      .astype(bool)
)
frontier["O_ever"] = frontier["O_operational"]
frontier["OS_ever"] = frontier["OS_semantic"]
frontier["OS_new_frontier"] = (~frontier["O_ever"]) & frontier["OS_ever"]
frontier["O_only_frontier"] = frontier["O_ever"] & (~frontier["OS_ever"])
frontier["both_ever"] = frontier["O_ever"] & frontier["OS_ever"]
frontier["neither_ever"] = (~frontier["O_ever"]) & (~frontier["OS_ever"])

frontier_summary = {
    "O_distinct_tasks_solved": int(frontier["O_ever"].sum()),
    "OS_distinct_tasks_solved": int(frontier["OS_ever"].sum()),
    "OS_expanded_frontier_tasks": int(frontier["OS_new_frontier"].sum()),
    "O_only_frontier_tasks": int(frontier["O_only_frontier"].sum()),
    "both_ever_tasks": int(frontier["both_ever"].sum()),
    "neither_ever_tasks": int(frontier["neither_ever"].sum()),
}
print(json.dumps(frontier_summary, indent=2))
display(frontier.reset_index())
frontier.reset_index().to_csv(STUDY_ROOT/"task_level_solved_frontier.csv", index=False)


### 7.3 Treatment uptake

Executed markers measure whether the requested check was invoked, not whether it was semantically meaningful.


In [ ]:
uptake = (
    ev.groupby("condition")
      .agg(
          runs=("task_id","count"),
          marker_executed=("behavior_marker_executed",lambda x:int(pd.Series(x).fillna(False).sum())),
          marker_calls=("behavior_marker_calls","sum"),
          post_marker_edits=("post_marker_edit",lambda x:int(pd.Series(x).fillna(False).sum())),
      )
      .reset_index()
)
uptake["marker_rate"] = uptake["marker_executed"] / uptake["runs"]
display(uptake)
uptake.to_csv(STUDY_ROOT/"treatment_uptake_summary.csv", index=False)


### 7.4 Resource use and patch sensitivity

Patch-production measures are post-treatment outcomes. Conditional comparisons among non-empty patches are descriptive, not causal.


In [ ]:
resource = (
    ev.groupby("condition")
      .agg(
          runs=("task_id","count"),
          resolved=("resolved_local","sum"),
          nonempty=("nonempty_patch","sum"),
          median_calls=("tool_calls","median"),
          median_runtime_s=("elapsed_seconds","median"),
          median_patch_chars_all=("patch_chars","median"),
      )
      .reset_index()
)

nonempty_medians=[]
for cond in CONDITIONS:
    x=ev[(ev.condition==cond)&(ev.nonempty_patch)]
    nonempty_medians.append({
        "condition":cond,
        "median_patch_chars_nonempty": float(x.patch_chars.median()) if len(x) else np.nan,
        "n_nonempty": len(x),
    })
resource = resource.merge(pd.DataFrame(nonempty_medians),on="condition",how="left")
display(resource)
resource.to_csv(STUDY_ROOT/"resource_and_patch_sensitivity.csv", index=False)

pchars = ev.pivot_table(index=["task_id","rep"], columns="condition", values="patch_chars", aggfunc="first")
both_nonempty = pchars[(pchars["O_operational"]>0)&(pchars["OS_semantic"]>0)].copy()
if len(both_nonempty):
    both_nonempty["delta_OS_minus_O"] = both_nonempty["OS_semantic"] - both_nonempty["O_operational"]
    patch_pair_summary = {
        "paired_nonempty_n": len(both_nonempty),
        "median_delta": float(both_nonempty["delta_OS_minus_O"].median()),
        "OS_smaller": int((both_nonempty["delta_OS_minus_O"]<0).sum()),
        "same_size": int((both_nonempty["delta_OS_minus_O"]==0).sum()),
        "OS_larger": int((both_nonempty["delta_OS_minus_O"]>0).sum()),
    }
else:
    patch_pair_summary = {"paired_nonempty_n":0}
print(json.dumps(patch_pair_summary, indent=2))
both_nonempty.reset_index().to_csv(STUDY_ROOT/"paired_nonempty_patch_sensitivity.csv", index=False)


### 7.5 Replication transitions

This separates stable successes/failures from repetition instability.


In [ ]:
repwide = ev.pivot_table(
    index=["task_id","condition"],
    columns="rep",
    values="resolved_local",
    aggfunc="first"
)
repwide.columns = [f"rep_{int(c)}" for c in repwide.columns]
repwide = repwide.reset_index()

if {"rep_1","rep_2"}.issubset(repwide.columns):
    def transition(r):
        a,b=int(bool(r["rep_1"])),int(bool(r["rep_2"]))
        return {(0,0):"fail->fail",(0,1):"fail->success",(1,0):"success->fail",(1,1):"success->success"}[(a,b)]
    repwide["transition"]=repwide.apply(transition,axis=1)

display(repwide)
repwide.to_csv(STUDY_ROOT/"replication_transition_table.csv",index=False)
if "transition" in repwide:
    display(repwide.groupby(["condition","transition"]).size().rename("n").reset_index())


### 7.6 Evaluator-status audit

Do not automatically interpret every unresolved run as a semantically incorrect repair.


In [ ]:
def evaluator_class(row):
    if bool(row.get("resolved_local",False)):
        return "resolved"
    ec = row.get("test_exit_code",None)
    err = str(row.get("error","") or "").lower()
    try:
        ec_num = float(ec)
    except Exception:
        ec_num = np.nan
    if ec_num == -1:
        return "verification_not_completed_or_blocked"
    if not np.isnan(ec_num):
        return "unresolved_with_evaluator_exit_cause_unverified"
    if any(k in err for k in ["timeout","tool","turn"]):
        return "verification_not_completed_or_blocked"
    return "unresolved_status_unclear"

ev["evaluator_outcome_class"] = ev.apply(evaluator_class,axis=1)
eval_audit = ev.groupby(["condition","evaluator_outcome_class"]).size().rename("n").reset_index()
display(eval_audit)
eval_audit.to_csv(STUDY_ROOT/"evaluator_outcome_summary.csv",index=False)

ev[[
    "task_id","rep","condition","resolved_local","test_exit_code","error",
    "patch_chars","tool_calls","elapsed_seconds","evaluator_outcome_class"
]].to_csv(STUDY_ROOT/"evaluator_outcome_audit.csv",index=False)


### 7.7 Manual semantic-quality audit template

For every O+S run, manually inspect whether the specification was referenced, whether a relevant check ran in the correct workspace, whether it tested required behavior or preserved invariants, whether it was discriminative, and whether the patch was revised afterward.


In [ ]:
manual = ev[ev.condition=="OS_semantic"][[
    "task_id","rep","condition","resolved_local","expected_marker",
    "behavior_marker_executed","post_marker_edit"
]].copy()

for col in [
    "specification_referenced",
    "check_invoked_and_relevant",
    "correct_workspace",
    "required_behavior_tested",
    "preserved_invariant_or_boundary_tested",
    "discriminative_check",
    "patch_revised_after_check",
    "manual_notes",
    "reviewer",
]:
    manual[col] = ""

manual_path = STUDY_ROOT/"manual_semantic_quality_audit_TEMPLATE.csv"
manual.to_csv(manual_path,index=False)
print("Manual audit template:",manual_path)
display(manual.head())


### 7.8 Paper-ready export

In [ ]:
paper_summary = {
    "primary": primary,
    "frontier": frontier_summary,
    "patch_sensitivity": patch_pair_summary,
    "spec_sha256": SPEC_SHA256,
    "cohort": {
        "evaluation_tasks": sorted(eval_ids),
        "n_tasks": len(eval_ids),
        "n_repetitions": N_REPS,
        "n_eval_runs": TOTAL_EVAL_RUNS,
    },
}
(STUDY_ROOT/"paper_ready_summary.json").write_text(json.dumps(paper_summary,indent=2)+"\n")

cols = [
    "task_id","rep","condition","resolved_local","nonempty_patch","patch_chars",
    "tool_calls","elapsed_seconds","behavior_marker_executed",
    "behavior_marker_calls","post_marker_edit","test_exit_code",
    "evaluator_outcome_class"
]
ev[cols].to_csv(STUDY_ROOT/"paper_ready_run_table.csv",index=False)
print(json.dumps(paper_summary,indent=2))


## 8. Export the complete private research bundle

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink

manifest={
    "study":"Gemma4_Operational_vs_Semantic_12x2x2",
    "fresh_issue_policy":"excludes all task IDs in preserved 416-trajectory corpus",
    "n_preflight_tasks":N_PREFLIGHT_TASKS,
    "n_evaluation_tasks":N_EVAL_TASKS,
    "n_repetitions":N_REPS,
    "planned_runs":TOTAL_RUNS,
    "completed_runs":len(results),
    "evaluation_runs":TOTAL_EVAL_RUNS,
    "baseline_zip_sha256":ORIGINAL_ALT_V1_SHA256,
    "cohort_sha256":hashlib.sha256((STUDY_ROOT/"cohort.json").read_bytes()).hexdigest(),
    "semantic_spec_sha256":SPEC_SHA256,
    "model":MODEL,
    "temperature":0.2,
    "thinking_budget":4096,
    "max_tool_calls":28,
    "max_turns":80,
    "max_time_minutes":8,
}
(STUDY_ROOT/"study_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")

tmp=FINAL_ZIP.with_suffix(".zip.tmp")
with ZipFile(tmp,"w",ZIP_DEFLATED,compresslevel=2) as z:
    for p in sorted(STUDY_ROOT.rglob("*")):
        if p.is_file() and not p.is_symlink():
            z.write(p,p.relative_to(WORKING_DIR).as_posix())
os.replace(tmp,FINAL_ZIP)

with ZipFile(FINAL_ZIP) as z:
    assert z.testzip() is None

print("Research bundle:",FINAL_ZIP)
print("Size MB:",round(FINAL_ZIP.stat().st_size/1e6,2))
display(FileLink(str(FINAL_ZIP)))


## 9. Outputs

The run writes `Gemma4_Operational_vs_Semantic_12x2x2_Research.zip` to `/kaggle/working/`.

Keep the filled `semantic_specs_frozen.csv` private: it contains issue text and may include competition-derived information.

The primary analysis is the paired **O+S minus O** resolution difference on the 12 evaluation issues, with both repetitions kept together in the task-cluster bootstrap.
